In [1]:
import torch
import gc

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  —  {p.total_memory / 1024**3:.1f} GB VRAM")
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU disponibili: {torch.cuda.device_count()}")

  [0] Tesla T4  —  14.6 GB VRAM
GPU disponibili: 2
  [1] Tesla T4  —  14.6 GB VRAM
GPU disponibili: 2


In [2]:
%%bash
pip install -q basicsr facexlib lpips einops timm scikit-image pyyaml tensorboard pyiqa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [3]:
import os, sys, shutil, yaml, glob, subprocess

# ─── MODIFICA QUESTI PATH ────────────────────────────────────────────────────
CODE_SRC  = "/kaggle/input/datasets/francescoardolino02/fcadreamuhdtrain/FcaDreamuhdTrain"
DATA_ROOT = "/kaggle/input/datasets/francescoardolino02/uhd-ll-crop-1024/uhd_ll_3k_crop"   # contiene training_set/ e testing_set/
# ─────────────────────────────────────────────────────────────────────────────

CODE_DEST = "/kaggle/working/FcaDreamuhd"
EXP_DIR   = "/kaggle/working/experiments"

if os.path.exists(CODE_DEST):
    shutil.rmtree(CODE_DEST)
shutil.copytree(CODE_SRC, CODE_DEST)
os.makedirs(EXP_DIR, exist_ok=True)
print(f"✅ Codice copiato in: {CODE_DEST}")
print(f"✅ Esperimenti in   : {EXP_DIR}")

✅ Codice copiato in: /kaggle/working/FcaDreamuhd
✅ Esperimenti in   : /kaggle/working/experiments


In [4]:
VAE_TRAINED = "/kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth"

import os
if not os.path.exists(VAE_TRAINED):
    raise FileNotFoundError(f"Errore: non trovo il VAE in {VAE_TRAINED}")
print(f"✅ VAE trainato trovato: {VAE_TRAINED}")

✅ VAE trainato trovato: /kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth


In [5]:
DREAM_YML_SRC  = os.path.join(CODE_DEST, "options/DreamUHD_LL.yml")

In [6]:
DREAM_YML_DEST = "/kaggle/working/DreamUHD_LL_kaggle.yml"

with open(DREAM_YML_SRC, "r") as f:
    dream_cfg = yaml.safe_load(f)

dream_cfg["name"] = "FcaDreamUHD_Stage2"

DREAM_NAME    = dream_cfg.get("name", "FcaDreamUHD")
DREAM_EXP_DIR = os.path.join(EXP_DIR, DREAM_NAME)
os.makedirs(DREAM_EXP_DIR, exist_ok=True)

# Dataset
dream_cfg["datasets"]["train"]["dataroot_gt"] = os.path.join(DATA_ROOT, "training_set/gt")
dream_cfg["datasets"]["train"]["dataroot_lq"] = os.path.join(DATA_ROOT, "training_set/input")
dream_cfg["datasets"]["val"]["dataroot_gt"]   = os.path.join(DATA_ROOT, "testing_set/gt")
dream_cfg["datasets"]["val"]["dataroot_lq"]   = os.path.join(DATA_ROOT, "testing_set/input")

# VAE prodotto nello Stage 1
dream_cfg["network_g"]["vae_weight"] = VAE_TRAINED
dream_cfg["network_g"]["config"]     = os.path.join(CODE_DEST, "options/VAE_LL.yml")

# ⚠️ Forza i path esperimenti in /kaggle/working/
dream_cfg["path"]["experiments_root"] = DREAM_EXP_DIR
dream_cfg["path"]["models"]           = os.path.join(DREAM_EXP_DIR, "models")
dream_cfg["path"]["training_states"]  = os.path.join(DREAM_EXP_DIR, "training_states")
dream_cfg["path"]["log"]              = DREAM_EXP_DIR
dream_cfg["path"]["visualization"]    = os.path.join(DREAM_EXP_DIR, "visualization")

# Configuriamo il resume nativo puntando alle cartelle ufficiali dell'esperimento
dream_cfg["path"]["resume_state"] = "/kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/training_states/15000.state"
dream_cfg["path"]["pretrain_network_g"] = None
dream_cfg["path"]["ignore_resume_networks"] = ["network_d"]

with open(DREAM_YML_DEST, "w") as f:
    yaml.dump(dream_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"✅ DreamUHD config: {DREAM_YML_DEST}")
print(f"   experiments_root : {dream_cfg['path']['experiments_root']}")
print(f"   vae_weight       : {dream_cfg['network_g']['vae_weight']}")
print(f"   resume_state     : {dream_cfg['path']['resume_state']}")

✅ DreamUHD config: /kaggle/working/DreamUHD_LL_kaggle.yml
   experiments_root : /kaggle/working/experiments/FcaDreamUHD_Stage2
   vae_weight       : /kaggle/input/datasets/francescoardolino02/experiments/experiments/VAE_LL/models/net_g_45000.pth
   resume_state     : /kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/training_states/15000.state


In [7]:
TRAIN_PY = "/kaggle/working/FcaDreamuhd/basicsr/train.py"

In [8]:
import os

dream_arch_path = "/kaggle/working/FcaDreamuhd/basicsr/archs/DreamUHD_arch.py"

with open(dream_arch_path, 'r') as f:
    codice = f.read()

codice = codice.replace('["vanilla","FE-block","FE-block2"]', '["vanilla","FE-block","FE-block2","FEblock"]')
codice = codice.replace('res_type == "FE-block":', 'res_type in ["FE-block", "FEblock"]:')

with open(dream_arch_path, 'w') as f:
    f.write(codice)

print("✅ Fix applicato a DreamUHD_arch.py! Ora puoi lanciare lo Stage 2.")

✅ Fix applicato a DreamUHD_arch.py! Ora puoi lanciare lo Stage 2.


In [9]:
"""
Questa cella serve poichè BasicSR ha una funzione di auto_resume che automaticamente va acercare il .state nella working
directory, quindi dobbiamo spostare i file .state e .pth da cui dovrà ripartire il modello nella wd.

Anche il file /kaggle/working/DreamUHD_LL_kaggle.yml
"""

import os, shutil

# Usiamo il path esatto che BasicSR sta pretendendo nel log dell'errore
dest_models_dir = "/kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/models"
dest_states_dir = "/kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/training_states"

# Creiamo le cartelle nel percorso preteso da BasicSR
os.makedirs(dest_models_dir, exist_ok=True)
os.makedirs(dest_states_dir, exist_ok=True)

# Path di origine (Kaggle Input)
src_pth   = "/kaggle/input/datasets/francescoardolino02/val-intermedi/net_g_15000.pth"
src_state = "/kaggle/input/datasets/francescoardolino02/val-intermedi/15000.state"

# Copia dei file
shutil.copy(src_pth,   os.path.join(dest_models_dir, "net_g_15000.pth"))
shutil.copy(src_state, os.path.join(dest_states_dir, "15000.state"))

# Copia dello YAML (Invariato)
yml_hardcoded_dir = "/kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2"
os.makedirs(yml_hardcoded_dir, exist_ok=True)
shutil.copy("/kaggle/working/DreamUHD_LL_kaggle.yml", f"{yml_hardcoded_dir}/DreamUHD_LL_kaggle.yml")

print(f"✅ Pesi .pth copiati FORZATAMENTE in: {dest_models_dir}/net_g_15000.pth")
print(f"✅ Stato .state copiato FORZATAMENTE in: {dest_states_dir}/15000.state")
print(f"✅ YAML copiato in: {yml_hardcoded_dir}")

✅ Pesi .pth copiati FORZATAMENTE in: /kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/models/net_g_15000.pth
✅ Stato .state copiato FORZATAMENTE in: /kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2/training_states/15000.state
✅ YAML copiato in: /kaggle/working/FcaDreamuhd/experiments/FcaDreamUHD_Stage2


In [10]:
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    TRAIN_PY,
    "-opt", DREAM_YML_DEST,
    "--launcher", "pytorch",
]
print("[STAGE 2] Comando:", " ".join(cmd))
print("─" * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    cwd=CODE_DEST,
    env={**os.environ, "PYTHONPATH": CODE_DEST},
)
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
print(f"\nExit code: {proc.returncode}")

[STAGE 2] Comando: torchrun --nproc_per_node=2 --master_port=29500 /kaggle/working/FcaDreamuhd/basicsr/train.py -opt /kaggle/working/DreamUHD_LL_kaggle.yml --launcher pytorch
──────────────────────────────────────────────────────────────────────


W0521 18:49:24.113000 73 torch/distributed/run.py:852] 


W0521 18:49:24.113000 73 torch/distributed/run.py:852] *****************************************


W0521 18:49:24.113000 73 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 


W0521 18:49:24.113000 73 torch/distributed/run.py:852] *****************************************


...

	 # ssim: 0.8737	Best: 0.8797 @ 30000 iter


2026-05-22 06:37:21,357 INFO: [FcaDr..][epoch: 32, iter:  33,100, lr:(6.746e-04,)] [eta: 18:10:01, time (data): 2.257 (0.009)] l_pix: 3.9872e-02 l_freq: 1.2026e+00 


2026-05-22 06:41:07,862 INFO: [FcaDr..][epoch: 32, iter:  33,200, lr:(6.738e-04,)] [eta: 18:05:54, time (data): 2.264 (0.009)] l_pix: 7.9709e-02 l_freq: 1.1902e+00 


2026-05-22 06:44:55,704 INFO: [FcaDr..][epoch: 32, iter:  33,300, lr:(6.731e-04,)] [eta: 18:01:50, time (data): 2.258 (0.009)] l_pix: 4.6314e-02 l_freq: 1.1815e+00 


2026-05-22 06:48:43,340 INFO: [FcaDr..][epoch: 32, iter:  33,400, lr:(6.723e-04,)] [eta: 17:57:46, time (data): 2.275 (0.009)] l_pix: 1.2353e-01 l_freq: 1.4345e+00 
